In [15]:
import os 
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import sqlalchemy
import psycopg2
import pandas as pd

load_dotenv()

HOST = os.getenv("POSTGIS_HOST")
PORT = int(os.getenv("POSTGIS_PORT"))
USER = os.getenv("POSTGIS_USER")
PASSWORD = os.getenv("POSTGIS_PASSWORD")

engine = create_engine(
    f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/incendies",
    connect_args={"options": "-csearch_path=incendies_schema,public"}
)

# Test de connexion
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version()"))
        print("Connexion réussie !")
        print("Version PostgreSQL :", result.scalar())

        dbs = conn.execute(
            text("SELECT datname FROM pg_database WHERE datistemplate = false ORDER BY datname")
        ).fetchall()
        print("Bases de données :", [db[0] for db in dbs])
except Exception as e:
    print("Erreur de connexion :", e)


Connexion réussie !
Version PostgreSQL : PostgreSQL 17.5 (Debian 17.5-1.pgdg110+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 10.2.1-6) 10.2.1 20210110, 64-bit
Bases de données : ['incendies', 'postgres']


In [3]:
DATABASE_NAME="incendies"

def get_engine():
    return create_engine(
        f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE_NAME}",
        connect_args={"options": "-csearch_path=incendies,public"},
    )

engine = get_engine()

## Création de la table commune_jour

In [ ]:
ddl = [
    """
    CREATE TABLE commune_jour AS
    SELECT 
        c.id_commune,
        d.jour::date AS date_jour,
        FALSE AS has_fire
    FROM 
        commune c
    CROSS JOIN 
        generate_series(
            '2006-01-01'::date, 
            '2025-12-31'::date, 
            '1 day'::interval
        ) AS d(jour);
    """
]

with engine.begin() as conn:
    conn.execute(text("SET search_path TO incendies, public"))
    for stmt in ddl:
        conn.execute(text(stmt))
        conn.commit()

In [ ]:
ddl = [
    """
    ALTER TABLE commune_jour
    ADD CONSTRAINT pk_commune_jour PRIMARY KEY (id_commune, date_jour);
    """
]

with engine.begin() as conn:
    conn.execute(text("SET search_path TO incendies, public"))
    for stmt in ddl:
        conn.execute(text(stmt))
        conn.commit()

In [ ]:
ddl = [
    """
    ALTER TABLE commune_jour 
    ALTER COLUMN has_fire TYPE SMALLINT 
    USING (has_fire::INT);
    """
]

with engine.begin() as conn:
    conn.execute(text("SET search_path TO incendies, public"))
    for stmt in ddl:
        conn.execute(text(stmt))
        conn.commit()

In [ ]:
sql = text("""
    UPDATE commune_jour cj
    SET has_fire = 1
    FROM incendie i
    JOIN commune c ON c.code_insee = i.code_insee
    WHERE cj.id_commune = c.id_commune 
      AND cj.date_jour = i.date_premiere_alerte::date;
""")

with engine.begin() as conn:
    conn.execute(sql)

### Feature engineering

In [5]:
ddl = [
    """
    ALTER TABLE commune_jour
    ADD COLUMN IF NOT EXISTS nb_incident_30j INT
    """,
    """
    ALTER TABLE commune_jour
    ADD COLUMN IF NOT EXISTS nb_incident_90j INT
    """,
    """
    ALTER TABLE commune_jour
    ADD COLUMN IF NOT EXISTS nb_incident_365j INT
    """,
    """
    ALTER TABLE commune_jour
    ADD COLUMN IF NOT EXISTS Surface_totale_5a INT
    """,
    """
    ALTER TABLE commune_jour
    ADD COLUMN IF NOT EXISTS buffer_10km INT
    """,
    """
    ALTER TABLE commune_jour
    ADD COLUMN IF NOT EXISTS buffer_20km INT
    """,
    """
    ALTER TABLE commune_jour
    ADD COLUMN IF NOT EXISTS buffer_50km INT
    """
]

with engine.connect() as conn:
    conn.execute(text("SET search_path TO incendies, public"))
    for stmt in ddl:
        conn.execute(text(stmt))
        conn.commit()

In [25]:
# récupération des dates du jour
dates = pd.read_sql(
    "SELECT DISTINCT date_jour FROM commune_jour ORDER BY date_jour",
    engine,
)

In [ ]:
test_engine = get_engine()

sql = text("""
    INSERT INTO commune_jour (
        id_commune, date_jour,
        nb_incendies_30j, nb_incendies_90j, nb_incendies_365j
    )
    SELECT
        c.id_commune,
        :date_jour,
        COUNT(*) FILTER (
            WHERE i.date_premiere_alerte::date < :date_jour + INTERVAL '30 days'
        ),
        COUNT(*) FILTER (
            WHERE i.date_premiere_alerte::date < :date_jour + INTERVAL '90 days'
        ),
        COUNT(*)
    FROM commune c
    JOIN incendie i ON i.code_insee = c.code_insee
        AND i.date_premiere_alerte::date >= :date_jour 
        AND i.date_premiere_alerte::date < :date_jour  + INTERVAL '365 days'
    GROUP BY c.id_commune
    -- UPSERT
    -- Si une ligne existe déjà pour cette commune à cette date, 
    -- on remplace ses colonnes nb_incendies_30, nb_incendies_90, nb_incendies_365 
    -- par les nouvelles valeurs calculées, plutôt que de générer une erreur ou de créer un doublon.
    ON CONFLICT (id_commune, date_jour) DO UPDATE
    SET nb_incendies_30j  = excluded.nb_incendies_30j,
        nb_incendies_90j  = excluded.nb_incendies_90j,
        nb_incendies_365j = excluded.nb_incendies_365j;
""")

with engine.connect() as conn:
    for jour in dates["date_jour"]:
        conn.execute(sql, {"date_jour": jour})
        conn.commit()

<>:12: SyntaxWarning: invalid escape sequence '\:'
<>:12: SyntaxWarning: invalid escape sequence '\:'
/tmp/ipykernel_22978/355422206.py:12: SyntaxWarning: invalid escape sequence '\:'
  WHERE i.date_premiere_alerte::date < :date_jour\:\:date + INTERVAL '30 days'


### Mise à jour de la colonne surface_totale_5a : script jamais testé.

In [ ]:
sql = text("""
    INSERT INTO commune_jour (
        id_commune, date_jour,
        surface_totale_5a
    )
    SELECT
        c.id_commune,
        :date_jour,
        SUM(surface_parcourue)
    FROM commune c
    JOIN incendie i ON i.code_insee = c.code_insee
        AND i.date_premiere_alerte::date >= :date_jour 
        AND i.date_premiere_alerte::date < :date_jour + INTERVAL '1825 days'
    GROUP BY c.id_commune
    ON CONFLICT (id_commune, date_jour) DO UPDATE
    SET surface_totale_5a  = excluded.surface_totale_5a;
""")

with engine.connect() as conn:
    for jour in dates["date_jour"]:
        conn.execute(sql, {"date_jour": jour})
        conn.commit()


## Mise à jour de la colonne buffer_10km : script jamais testé.

### deux versions : la première est optimisée, mais 
- on ré

In [ ]:
sql = text("""
    WITH paires_communes AS (
        SELECT 
            c1.id_commune AS ref_commune,
            c2.code_insee AS search_code_insee
        FROM commune c1
        JOIN localisation l1 ON l1.id_localisation = c1.localisation
        JOIN localisation l2 ON ST_DWithin(
            ST_Transform(l1.geom, 2154),
            ST_Transform(l2.geom, 2154),
            10000
        )
        JOIN commune c2 ON c2.localisation = l2.id_localisation
    )
    SELECT 
        p.ref_commune,
        COUNT(i.id_incendie) AS total_incendies
    FROM paires_communes p
    LEFT JOIN incendie i ON i.code_insee = p.search_code_insee
    GROUP BY p.ref_commune;
""")

